# EDA — Precipitation and Rainfall-Runoff Relationship
## Itajaí-Açu Basin | INMET + CHIRPS

---

### What this notebook answers

1. Are the INMET stations representative of the entire basin, or are they too sparse?
2. Does CHIRPS agree with the INMET stations? Where does it diverge?
3. **What is the lag between the rainfall peak and the streamflow peak in Blumenau?**
   ← This answer defines the `hindcast_length` in the training YAML.
4. Does the rainfall-runoff relationship vary by season (saturated vs. dry soil)?

### Why do this before training?

The `hindcast_length` (historical context window of the LSTM) should be
at least as long as the characteristic rainfall-runoff lag of the basin.
For the Itajaí-Açu, with ~15,000 km², the estimated concentration time
is 24–48 h for sub-basins and up to 72 h for the full basin above
Blumenau. If the current config uses 168 h (7 days), that should be sufficient
— but let's confirm empirically.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from scipy import signal, stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

FIGDIR = Path('../../reports/figures')
FIGDIR.mkdir(parents=True, exist_ok=True)

RAW_DIR    = Path('../../data/raw')
PRECIP_DIR = RAW_DIR / 'precipitation'
FLOW_DIR   = RAW_DIR / 'streamflow'

STATION_ANA = '83500000'

FLOOD_EVENTS = {
    '1983-11-09': 'Nov/1983',
    '2008-11-23': 'Nov/2008',
    '2011-09-08': 'Sep/2011',
}

print('Environment configured.')

## 1. Download and Loading of Data

In [ ]:
from src.data.ana_downloader import download_series, save_raw
from src.data.precipitation_downloader import (
    download_all_basin_stations,
    download_chirps_range,
    chirps_basin_average,
    list_inmet_stations_in_basin,
)

# ── Streamflow ─────────────────────────────────────────────────────────────────────
vazao_path = FLOW_DIR / f'{STATION_ANA}_vazao_raw.parquet'
if not vazao_path.exists():
    df_q = download_series(STATION_ANA, 'vazao', start_year=1981)
    save_raw(df_q, STATION_ANA, 'vazao')
else:
    df_q = pd.read_parquet(vazao_path)

Q = df_q['value'].rename('Q_m3s')

# ── INMET Precipitation ────────────────────────────────────────────────────────
inmet_files = list(PRECIP_DIR.glob('inmet_*_daily.parquet'))
if not inmet_files:
    print('Downloading INMET stations from the basin...')
    inmet_data = download_all_basin_stations(start_date='2000-01-01', out_dir=PRECIP_DIR)
    inmet_files = list(PRECIP_DIR.glob('inmet_*_daily.parquet'))
else:
    print(f'{len(inmet_files)} INMET station(s) already downloaded.')

# Load all INMET stations into a single DataFrame
inmet_dfs = {}
for f in inmet_files:
    sid = f.stem.replace('inmet_', '').replace('_daily', '')
    inmet_dfs[sid] = pd.read_parquet(f)['precip_mm']

if inmet_dfs:
    df_inmet = pd.DataFrame(inmet_dfs)
    df_inmet.index = pd.to_datetime(df_inmet.index)
    print(f'INMET: {len(df_inmet.columns)} stations, {df_inmet.index.min().date()} → {df_inmet.index.max().date()}')
else:
    df_inmet = pd.DataFrame()
    print('No INMET station available locally.')

# ── CHIRPS Precipitation ───────────────────────────────────────────────────────
chirps_files = list(PRECIP_DIR.glob('chirps_*_itajai.nc'))
if not chirps_files:
    print('Downloading CHIRPS... (may take a few minutes)')
    chirps_files = download_chirps_range(start_year=1981, out_dir=PRECIP_DIR)
else:
    print(f'{len(chirps_files)} CHIRPS file(s) already downloaded.')

if chirps_files:
    df_chirps = chirps_basin_average(PRECIP_DIR)
    print(f'CHIRPS: {df_chirps.index.min().date()} → {df_chirps.index.max().date()}')
else:
    df_chirps = pd.DataFrame()
    print('No CHIRPS data available.')

print('\nData loaded.')

## 2. Map of INMET Stations in the Basin

Visualize the spatial distribution before any analysis.

**What to look for:** stations covering the headwaters of the three main
tributaries (Itajaí do Sul, Itajaí do Oeste, Itajaí do Norte) and the
coastal region near Blumenau. Stations concentrated only near the outlet
under-represent rainfall from the headwaters — which is where the large events
originate (orographic rainfall on the Serra).

In [ ]:
try:
    stations = list_inmet_stations_in_basin()

    fig, ax = plt.subplots(figsize=(9, 7))

    if not stations.empty:
        sc = ax.scatter(
            stations['lon'], stations['lat'],
            c='steelblue', s=80, zorder=5, label='INMET automatic stations'
        )
        for _, row in stations.iterrows():
            ax.annotate(
                row.get('station_id', ''),
                (row['lon'], row['lat']),
                textcoords='offset points', xytext=(5, 3),
                fontsize=7
            )

    # Mark the Blumenau stream gauge station
    ax.scatter(-49.065, -26.915, c='crimson', s=120, marker='v', zorder=6,
               label='ANA Station 83500000 (Blumenau)')

    # Basin bounding box
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle(
        (-50.2, -27.6), 50.2 - 48.6, 27.6 - 26.5,
        fill=False, edgecolor='gray', lw=1.5, ls='--', label='Basin bounding box'
    ))

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('INMET Stations — Itajaí-Açu Basin', fontsize=12)
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGDIR / '09_mapa_estacoes_inmet.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Stations in the basin: {len(stations)}')
    if not stations.empty:
        print(stations[['station_id', 'name', 'lat', 'lon', 'start_date']].to_string())

except Exception as e:
    print(f'INMET API unavailable: {e}')
    print('Proceeding with CHIRPS analysis.')

## 3. INMET Station Completeness

Same heatmap analysis as in the streamflow notebook, but for the rain gauges.

**Expected result and possible issue:**
Automatic INMET stations have coverage from ~2000, but with frequent gaps
(sensor maintenance, transmission failure, batteries).
Gaps in rain gauge stations are more problematic than in streamflow because
rainfall is more intermittent — a NaN day could be a missing value
or a day with zero rainfall. The preprocessor must handle these cases
differently.

In [ ]:
if df_inmet.empty:
    print('INMET data not available — skipping section 3.')
else:
    completeness = (
        df_inmet.resample('MS').apply(lambda s: s.notna().mean())
        .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
    )

    fig, axes = plt.subplots(1, len(df_inmet.columns), figsize=(5 * len(df_inmet.columns), 8),
                              sharey=True)
    if len(df_inmet.columns) == 1:
        axes = [axes]

    month_names = ['J','F','M','A','M','J','J','A','S','O','N','D']

    for ax, station_id in zip(axes, df_inmet.columns):
        pivot = (
            df_inmet[[station_id]]
            .resample('MS').apply(lambda s: s.notna().mean())
            .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
            .pivot(index='year', columns='month', values=station_id)
        )
        pivot.columns = month_names
        sns.heatmap(pivot, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
                    linewidths=0.3, cbar=False)
        ax.set_title(f'St. {station_id}', fontsize=9)
        ax.set_xlabel('')

    fig.suptitle('INMET Completeness by Station (%)', fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGDIR / '10_completude_inmet.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Coverage statistics
    print('Completeness by station:')
    print(df_inmet.notna().mean().mul(100).round(1).to_string())

## 4. INMET × CHIRPS Comparison

**Why is this comparison critical?**

If INMET and CHIRPS agree well, we can use CHIRPS to fill
the periods without INMET coverage (before 2000) and have a
consistent series from 1981 to the present.

If they diverge systematically, we need to understand *why*:
- **CHIRPS overestimates intense rainfall?** Common in fast convective
  systems (CHIRPS interpolates and may smooth extreme peaks)
- **INMET is at an atypical location?** (sheltered valley, urban area)
- **Seasonal bias?** Does the bias change between summer and winter?

The answer determines whether we use CHIRPS directly or apply
bias correction before training the model.

In [ ]:
if df_inmet.empty or df_chirps.empty:
    print('Insufficient data for comparison — skipping section 4.')
else:
    # Average of INMET stations as a proxy for the observed areal mean
    inmet_mean = df_inmet.mean(axis=1).rename('INMET_mean')
    chirps_col = df_chirps.columns[0]
    chirps = df_chirps[chirps_col].rename('CHIRPS')

    # Align periods
    common = pd.concat([inmet_mean, chirps], axis=1).dropna()
    print(f'Common period for comparison: {common.index.min().date()} → {common.index.max().date()} ({len(common)} days)')

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Scatter
    axes[0].scatter(common['INMET_mean'], common['CHIRPS'], s=3, alpha=0.4, color='steelblue')
    lim = max(common.max())
    axes[0].plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
    axes[0].set_xlabel('INMET mean (mm/day)')
    axes[0].set_ylabel('CHIRPS (mm/day)')
    axes[0].set_title('Scatter INMET × CHIRPS')
    r, p = stats.pearsonr(common['INMET_mean'], common['CHIRPS'])
    axes[0].text(0.05, 0.95, f'r = {r:.3f}', transform=axes[0].transAxes,
                 va='top', fontsize=10, bbox=dict(facecolor='white', alpha=0.7))
    axes[0].legend()

    # Monthly bias
    monthly_bias = (
        (common['CHIRPS'] - common['INMET_mean'])
        .groupby(common.index.month)
        .mean()
    )
    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    axes[1].bar(month_names, monthly_bias.values, color=['salmon' if v > 0 else 'steelblue' for v in monthly_bias])
    axes[1].axhline(0, color='k', lw=0.8)
    axes[1].set_ylabel('Bias CHIRPS - INMET (mm/day)')
    axes[1].set_title('Monthly Bias')

    # Annual accumulation
    annual = common.resample('YE').sum()
    axes[2].plot(annual.index.year, annual['INMET_mean'], marker='o', ms=4, label='INMET', color='steelblue')
    axes[2].plot(annual.index.year, annual['CHIRPS'], marker='s', ms=4, label='CHIRPS', color='darkorange')
    axes[2].set_ylabel('Annual precipitation (mm)')
    axes[2].set_title('Annual Accumulation')
    axes[2].legend()

    fig.tight_layout()
    fig.savefig(FIGDIR / '11_inmet_vs_chirps.png', dpi=150, bbox_inches='tight')
    plt.show()

    bias_overall = (common['CHIRPS'] - common['INMET_mean']).mean()
    print(f'\nOverall bias CHIRPS - INMET: {bias_overall:+.2f} mm/day ({bias_overall/common["INMET_mean"].mean()*100:+.0f}%)')
    print(f'Pearson correlation:         {r:.3f}')

## 5. Rainfall → Streamflow Lag Analysis

**This is the most important analysis in this notebook.**

### How does cross-correlation work?

We compute the correlation between the precipitation series at `t` and the
streamflow series at `t + lag` for lags from 0 to 14 days. The lag with
the highest correlation is the "typical response time" of the basin.

**Physical-hydrological interpretation:**
- **Lag 0–1 days:** direct runoff (surface runoff) — intense rainfall
  on saturated or impervious soil
- **Lag 2–4 days:** subsurface flow (lateral flow)
- **Lag > 5 days:** contribution from shallow aquifers (baseflow)

For Blumenau, we expect a dominant lag of 1–3 days given the basin size
(~15,000 km²) and the topography of the Serra Geral, which accelerates
surface runoff in intense events.

**How this defines `hindcast_length`:**  
The hindcast_length should be ≥ max_lag * 2 to ensure the LSTM
sees the entire rainfall event before making a forecast.  
If the maximum lag is 5 days → hindcast_length ≥ 10 days.  
The current config with 168h (7 days) may need adjustment.

In [ ]:
# Choose the best available precipitation series
if not df_chirps.empty:
    P = df_chirps.iloc[:, 0].rename('P_mm')
    precip_source = 'CHIRPS'
elif not df_inmet.empty:
    P = df_inmet.mean(axis=1).rename('P_mm')
    precip_source = 'INMET (station average)'
else:
    print('No precipitation series available — skipping lag analysis.')
    P = None

if P is not None:
    # Align series
    df_pq = pd.concat([P, Q], axis=1).dropna()
    print(f'Common P×Q period: {df_pq.index.min().date()} → {df_pq.index.max().date()}')
    print(f'Precipitation source: {precip_source}')

    # ── Cross-correlation with scipy ─────────────────────────────────────────────
    #
    # Why use scipy.signal.correlate instead of pd.DataFrame.corr?
    # pd.corr computes a static correlation between two columns. For lag
    # analysis we need the correlation as a function of time offset,
    # which signal.correlate does in a vectorized and efficient manner.
    # We normalize by the number of samples and standard deviations to obtain
    # the Pearson coefficient at each lag.
    #
    max_lag = 14  # days

    p_std = (df_pq['P_mm'] - df_pq['P_mm'].mean()) / df_pq['P_mm'].std()
    q_std = (df_pq['Q_m3s'] - df_pq['Q_m3s'].mean()) / df_pq['Q_m3s'].std()

    xcorr = signal.correlate(q_std.values, p_std.values, mode='full') / len(df_pq)
    lags = signal.correlation_lags(len(q_std), len(p_std), mode='full')

    # Filter lags from 0 to max_lag (P precedes Q)
    mask = (lags >= 0) & (lags <= max_lag)
    lag_vals = lags[mask]
    corr_vals = xcorr[mask]

    best_lag = lag_vals[np.argmax(corr_vals)]
    best_corr = corr_vals.max()

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Cross-correlogram
    axes[0].bar(lag_vals, corr_vals, color='steelblue', alpha=0.8)
    axes[0].axvline(best_lag, color='crimson', lw=2, ls='--',
                    label=f'Maximum lag = {best_lag} days (r = {best_corr:.3f})')
    axes[0].set_xlabel('Lag (days) — P precedes Q')
    axes[0].set_ylabel('Correlation Coefficient')
    axes[0].set_title('Cross-Correlation P(t) × Q(t + lag)', fontsize=11)
    axes[0].legend(fontsize=9)
    axes[0].set_xticks(range(0, max_lag + 1))

    # Scatter P × Q with applied lag
    p_lagged = df_pq['P_mm'].shift(best_lag)
    valid = pd.concat([p_lagged, df_pq['Q_m3s']], axis=1).dropna()
    axes[1].scatter(valid['P_mm'], valid['Q_m3s'], s=3, alpha=0.3, color='steelblue')
    axes[1].set_xlabel(f'P(t - {best_lag}d) [mm/day]')
    axes[1].set_ylabel('Q(t) [m³/s]')
    axes[1].set_yscale('log')
    axes[1].set_title(f'Scatter P-Q with Lag = {best_lag} days')

    r2, _ = stats.pearsonr(valid['P_mm'], np.log1p(valid['Q_m3s']))
    axes[1].text(0.97, 0.05, f'r(P, logQ) = {r2:.3f}',
                 transform=axes[1].transAxes, ha='right', fontsize=9,
                 bbox=dict(facecolor='white', alpha=0.7))

    fig.tight_layout()
    fig.savefig(FIGDIR / '12_lag_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n=== LAG ANALYSIS RESULT ===')
    print(f'Lag of maximum correlation:  {best_lag} days')
    print(f'Correlation at maximum lag:  {best_corr:.3f}')
    print(f'\nImplication for YAML config:')
    recommended_hindcast = max(best_lag * 2, 7)
    print(f'  Recommended hindcast_length: ≥ {recommended_hindcast} days ({recommended_hindcast*24} h)')

## 6. Lag Analysis by Season

The hydrological response of the basin **is not constant throughout the year**.

- **Saturated soil (summer/autumn):** small rainfall events generate large floods
  (high runoff ratio, short lag)
- **Dry soil (winter/spring):** a large portion of rainfall is absorbed
  before generating runoff (low runoff ratio, long lag)

The LSTM model captures this implicitly via the hidden state (soil
memory). But confirming that the pattern exists in the data justifies why the model
needs a long enough hindcast to "remember" the antecedent
soil moisture conditions.

In [ ]:
if P is not None and len(df_pq) > 100:
    seasons = {
        'Summer (Dec-Feb)': [12, 1, 2],
        'Autumn (Mar-May)': [3, 4, 5],
        'Winter (Jun-Aug)': [6, 7, 8],
        'Spring (Sep-Nov)': [9, 10, 11],
    }

    fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=True)

    for ax, (season_name, months) in zip(axes, seasons.items()):
        subset = df_pq[df_pq.index.month.isin(months)]
        if len(subset) < 30:
            ax.set_title(f'{season_name}\n(insufficient data)')
            continue

        p_s = (subset['P_mm'] - subset['P_mm'].mean()) / (subset['P_mm'].std() + 1e-6)
        q_s = (subset['Q_m3s'] - subset['Q_m3s'].mean()) / (subset['Q_m3s'].std() + 1e-6)

        xcorr_s = signal.correlate(q_s.values, p_s.values, mode='full') / len(subset)
        lags_s = signal.correlation_lags(len(q_s), len(p_s), mode='full')
        mask_s = (lags_s >= 0) & (lags_s <= 14)

        best_s = lags_s[mask_s][np.argmax(xcorr_s[mask_s])]
        ax.bar(lags_s[mask_s], xcorr_s[mask_s], color='steelblue', alpha=0.8)
        ax.axvline(best_s, color='crimson', lw=2, ls='--')
        ax.set_title(f'{season_name}\nMax lag = {best_s} days', fontsize=9)
        ax.set_xlabel('Lag (days)')
        if ax == axes[0]:
            ax.set_ylabel('Correlation')

    fig.suptitle('Cross-Correlation P×Q by Season', fontsize=12, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGDIR / '13_lag_by_season.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Insufficient data for seasonal lag analysis.')

## 7. Event Analysis: Rainfall → Flood

Zoom into the three most severe historical events.
For each event we visualize the dual P + Q hydrograph,
allowing us to identify visually:
- When and where the rainfall began relative to the streamflow peak
- Whether there was accumulated rainfall for days before the peak (pre-saturated soil)
- The recession time after the peak

**For the model:** events with pre-saturation are the most dangerous and the
hardest to forecast. The LSTM needs the hindcast to detect the
antecedent condition; the forecast precipitation (forecast input) determines
the additional volume.

In [ ]:
if P is None:
    print('No precipitation series available — skipping section 7.')
else:
    WINDOW = 20  # days before and after the peak

    fig, axes = plt.subplots(len(FLOOD_EVENTS), 1,
                              figsize=(14, 5.5 * len(FLOOD_EVENTS)),
                              sharex=False)

    for ax, (date_str, label) in zip(axes, FLOOD_EVENTS.items()):
        peak = pd.Timestamp(date_str)
        t0 = peak - pd.Timedelta(days=WINDOW)
        t1 = peak + pd.Timedelta(days=WINDOW)

        q_evt = Q.loc[t0:t1]
        p_evt = P.loc[t0:t1] if not P.empty else pd.Series(dtype=float)

        # Primary axis: streamflow
        color_q = 'steelblue'
        ax.plot(q_evt.index, q_evt.values, color=color_q, lw=2, label='Streamflow (m³/s)')
        ax.fill_between(q_evt.index, q_evt.values, alpha=0.15, color=color_q)
        ax.axvline(peak, color='crimson', lw=1.5, ls='--', alpha=0.7, label='Declared peak')
        ax.set_ylabel('Streamflow (m³/s)', color=color_q)
        ax.tick_params(axis='y', labelcolor=color_q)

        # Secondary axis: precipitation (inverted bars at the top)
        if not p_evt.empty:
            ax2 = ax.twinx()
            ax2.bar(p_evt.index, p_evt.values, color='darkgreen', alpha=0.5,
                    width=0.8, label='Precipitation (mm/day)')
            ax2.set_ylabel('Precipitation (mm/day)', color='darkgreen')
            ax2.tick_params(axis='y', labelcolor='darkgreen')
            ax2.invert_yaxis()  # precipitation grows downward (meteorological convention)
            ax2.legend(loc='lower right', fontsize=8)

        ax.set_title(f'{label} | {date_str}', fontsize=11, fontweight='bold')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
        ax.legend(loc='upper left', fontsize=8)

        # Event statistics
        if not p_evt.empty and not q_evt.empty:
            total_p = p_evt.sum()
            max_q = q_evt.max()
            max_p = p_evt.max()
            ax.text(0.98, 0.97,
                    f'Total P: {total_p:.0f} mm\nMax P:   {max_p:.0f} mm/day\nMax Q:   {max_q:.0f} m³/s',
                    transform=ax.transAxes, ha='right', va='top', fontsize=9,
                    bbox=dict(facecolor='white', alpha=0.8))

    fig.suptitle('Hydrographs and Precipitation — Historical Events\nItajaí-Açu River / Blumenau',
                 fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(FIGDIR / '14_eventos_pq.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Runoff Ratio — Seasonality

The **runoff ratio** (Q / P) quantifies what fraction of rainfall becomes runoff.

- High in summer/autumn (saturated soil, vegetation with high ET in the wet season)
- Low in winter (groundwater recharge, low ET)

This pattern is fundamental for the model: the same rainfall volume
can generate very different floods depending on antecedent conditions.
The LSTM learns this via the hidden state (long-term memory),
but it helps to validate that the signal is present in the data.

In [ ]:
if P is not None and len(df_pq) > 365:
    # Monthly aggregates for monthly runoff ratio
    Q_monthly = df_pq['Q_m3s'].resample('MS').mean() * 86400 / 1000  # m³/s → mm/day (approx)
    P_monthly = df_pq['P_mm'].resample('MS').sum()

    rr = (Q_monthly / P_monthly.replace(0, np.nan)).clip(0, 3)
    rr_by_month = rr.groupby(rr.index.month).median()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    colors = ['salmon' if v > rr_by_month.median() else 'steelblue' for v in rr_by_month.values]
    axes[0].bar(month_names, rr_by_month.values, color=colors, alpha=0.85)
    axes[0].axhline(rr_by_month.median(), color='k', ls='--', lw=1)
    axes[0].set_ylabel('Runoff Ratio (Q/P)')
    axes[0].set_title('Median Monthly Runoff Ratio', fontsize=11)
    axes[0].set_ylim(0, rr_by_month.max() * 1.3)

    # Annual runoff ratio time series
    Q_annual = df_pq['Q_m3s'].resample('YE').mean() * 86400 * 365 / 1e9  # km³/year
    P_annual = df_pq['P_mm'].resample('YE').sum() / 1000  # m/year
    rr_annual = (Q_annual / (P_annual * 15000)).clip(0, 3)  # /basin area in km²

    axes[1].plot(rr_annual.index.year, rr_annual.values, marker='o', ms=5, color='steelblue')
    axes[1].axhline(rr_annual.mean(), color='k', ls='--', lw=1, label=f'Mean = {rr_annual.mean():.2f}')
    axes[1].set_ylabel('Annual Runoff Ratio')
    axes[1].set_title('Temporal Evolution of Runoff Ratio')
    axes[1].legend(fontsize=9)

    fig.tight_layout()
    fig.savefig(FIGDIR / '15_runoff_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Median Runoff Ratio by month:')
    print(dict(zip(month_names, rr_by_month.values.round(2))))
else:
    print('Insufficient data for runoff ratio analysis.')

## 9. Summary — Decisions for the Pipeline

Based on the analyses above, we document the decisions for the next steps:

In [ ]:
decisions = {
    'precipitation_source_training': {
        'decision': 'CHIRPS (1981–present) as the primary series; INMET as validation',
        'rationale': 'CHIRPS covers the entire historical period, including the 1983 '
                     'and 2008 events without gaps. INMET only covers 2000–present. '
                     'If CHIRPS-INMET bias is > 15%, apply bias correction before training.',
    },
    'hindcast_length': {
        'decision': 'Confirm via lag analysis — minimum 2x maximum lag',
        'rationale': 'Basin of ~15,000 km² with steep topography has an estimated lag '
                     'of 2–4 days. Current config of 168h (7 days) should be sufficient, '
                     'but lag analysis may suggest revision to 14+ days.',
    },
    'spatial_representation': {
        'decision': 'CHIRPS areal average as the sole precipitation feature in v1',
        'rationale': 'The base OpenHydroNet accepts a single precipitation feature per basin. '
                     'A future version may include multiple pixels or features of '
                     'spatial variability (std of basin pixels).',
    },
    'next_step': {
        'decision': 'notebooks/02_preprocessing: run caravan_formatter.py and validate NC',
        'rationale': 'Before training, visually validate the NetCDF: correct dates, '
                     'streamflow in m³/s without outliers, precipitation aligned.',
    },
}

for key, val in decisions.items():
    print(f'\n── {key} ──')
    print(f'  Decision:  {val["decision"]}')
    print(f'  Rationale: {val["rationale"]}')